In [21]:
#case study
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import prod


warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42

In [22]:
#loading breast cancer data
from sklearn.datasets import load_breast_cancer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

df = load_breast_cancer(as_frame=True)
X = df.data
y = df.target
print(X.shape)
print(y.shape)
print("Missing values:", X.isna().sum().sum())
print("Duplicate rows:", X.duplicated().sum())
print("Missing values:", y.isna().sum().sum())
print(y.value_counts())


(569, 30)
(569,)
Missing values: 0
Duplicate rows: 0
Missing values: 0
target
1    357
0    212
Name: count, dtype: int64


In [23]:
#train_test_split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

In [ ]:
#multimodel configuration on raw data

models_raw00 = {'knn': KNeighborsClassifier(), 
              'dt': DecisionTreeClassifier(), 
              'logistic': LogisticRegression(), 
              'rf': RandomForestClassifier(), 
              'grb': GradientBoostingClassifier(),
              'svc': SVC(probability=True),
              'xgbt':XGBClassifier()
              }

In [ ]:
#model evaluation_accuracy-scores
scores_raw00 = { }

for model_names, model in models_raw00.items():
    model.fit(X_train, y_train)
    scores_raw00[(model_names, 'train')]=model.score(X_train, y_train)
    scores_raw00[(model_names, 'test')]=model.score(X_test, y_test)

#results
results= pd.Series(scores_raw00).unstack()[['train', 'test']]
results = results.sort_values('test', ascending=True).round(4)
results

,train,test
dt,1.000000,0.903509
knn,0.947253,0.912281
svc,0.918681,0.929825
grb,1.000000,0.956140
logistic,0.945055,0.956140
rf,1.000000,0.956140
xgbt,1.000000,0.956140


In [ ]:
#model evaluation_roc_auc_scores
scores_raw01 ={ }

for model_names, model in models_raw00.items():
    model.fit(X_train, y_train)
    scores_raw01[(model_names, 'train')]=roc_auc_score(y_train, model.predict_proba(X_train)[:,1])
    scores_raw01[(model_names, 'test')]=roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
#results01
results01= pd.Series(scores_raw01).unstack()[['train', 'test']]
results01= results01.sort_values('test', ascending=True).round(4)
results01


,train,test
dt,1.000000,0.910714
knn,0.991197,0.955853
svc,0.974180,0.969577
grb,1.000000,0.990741
rf,1.000000,0.992560
xgbt,1.000000,0.992725
logistic,0.992178,0.993056


In [ ]:
#model eval_f1_scores
scores_raw02 = {}

for model_names, model in models_raw00.items():
    model.fit(X_train, y_train)

    scores_raw02[(model_names, 'train')] = f1_score(y_train, model.predict(X_train))
    scores_raw02[(model_names, 'test')] = f1_score(y_test, model.predict(X_test))

results02 = pd.Series(scores_raw02).unstack()[['train', 'test']]
results02 = results02.sort_values('test', ascending=True).roud(4)
results02

,train,test
dt,1.000000,0.928571
knn,0.958333,0.929577
svc,0.937815,0.945946
rf,1.000000,0.958333
grb,1.000000,0.965986
logistic,0.956522,0.965986
xgbt,1.000000,0.965986


In [36]:
#one code for all three scores
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

rows = {}
for name, model in models_raw00.items():
    pred = model.predict(X_test)
    score = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test)
    rows[name] = {
        'accuracy': accuracy_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'roc_auc': roc_auc_score(y_test, score),
    }

pd.DataFrame(rows).T.sort_values('f1', ascending=True).round(4)


,accuracy,f1,roc_auc
dt,0.9123,0.9286,0.9157
knn,0.9123,0.9296,0.9559
svc,0.9298,0.9459,0.9696
rf,0.9474,0.9583,0.9937
logistic,0.9561,0.9660,0.9931
grb,0.9561,0.9660,0.9907
xgbt,0.9561,0.9660,0.9927


In [24]:
########
#StandardScaler
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_std = sc.fit_transform(X_train)
X_test_std = sc.fit_transform(X_test)


In [ ]:
#models

models_std00 = {'knn': KNeighborsClassifier(), 
              'dt': DecisionTreeClassifier(), 
              'logistic': LogisticRegression(), 
              'rf': RandomForestClassifier(), 
              'grb': GradientBoostingClassifier(),
              'svc': SVC(probability=True),
              'xgbt':XGBClassifier()
              }

In [41]:
#model evaluation_accuracy-scores
scores_std00 = { }

for model_names, model in models_std00.items():
    model.fit(X_train_std, y_train)
    scores_std00[(model_names, 'train')]=model.score(X_train_std, y_train)
    scores_std00[(model_names, 'test')]=model.score(X_test_std, y_test)

#results
results= pd.Series(scores_std00).unstack()[['train', 'test']]
results = results.sort_values('test', ascending=True).round(4)
results

,train,test
dt,1.0000,0.9211
grb,1.0000,0.9561
rf,1.0000,0.9561
xgbt,1.0000,0.9561
knn,0.9736,0.9649
logistic,0.9890,0.9825
svc,0.9824,0.9825


In [42]:
#model evaluation_roc_auc_scores
scores_std001 ={ }

for model_names, model in models_std00.items():
    model.fit(X_train_std, y_train)
    scores_std001[(model_names, 'train')]=roc_auc_score(y_train, model.predict_proba(X_train_std)[:,1])
    scores_std001[(model_names, 'test')]=roc_auc_score(y_test, model.predict_proba(X_test_std)[:,1])
#results01
results001= pd.Series(scores_std001).unstack()[['train', 'test']]
results001= results001.sort_values('test', ascending=True).round(4)
results001


,train,test
dt,1.0000,0.9137
knn,0.9981,0.9792
grb,1.0000,0.9907
rf,1.0000,0.9916
xgbt,1.0000,0.9927
svc,0.9980,0.9950
logistic,0.9974,0.9954


In [ ]:
#model eval_f1_scores
scores_std002 = {}

for model_names, model in models_std00.items():
    model.fit(X_train_std, y_train)

    scores_std002[(model_names, 'train')] = f1_score(y_train, model.predict(X_train_std))
    scores_std002[(model_names, 'test')] = f1_score(y_test, model.predict(X_test_std))

results002 = pd.Series(scores_std002).unstack()[['train', 'test']]
results002 = results002.sort_values('test', ascending=True).round(4)
results002

,train,test
dt,1.0000,0.9220
rf,1.0000,0.9583
grb,1.0000,0.9660
xgbt,1.0000,0.9660
knn,0.9792,0.9726
logistic,0.9913,0.9861
svc,0.9861,0.9861


In [45]:
#one code for all three scores
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

rows = {}
for name, model in models_std00.items():
    pred = model.predict(X_test_std)
    score = model.predict_proba(X_test_std)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test_std)
    rows[name] = {
        'accuracy': accuracy_score(y_test, pred),
        'f1': f1_score(y_test, pred),
        'roc_auc': roc_auc_score(y_test, score),
    }

pd.DataFrame(rows).T.sort_values('f1', ascending=True).round(4)


,accuracy,f1,roc_auc
dt,0.9035,0.9220,0.9038
rf,0.9474,0.9583,0.9942
grb,0.9561,0.9660,0.9907
xgbt,0.9561,0.9660,0.9927
knn,0.9649,0.9726,0.9792
logistic,0.9825,0.9861,0.9954
svc,0.9825,0.9861,0.9950


In [67]:
#############reusable function
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def compute_metrics(model, X_, y_):
    pred  = model.predict(X_)
    proba = model.predict_proba(X_)
    return {
        "accuracy": accuracy_score(y_, pred),
        "f1_macro": f1_score(y_, pred, average="macro"),
        "auc_ovr_macro": (roc_auc_score(y_, proba[:, 1]) if proba.shape[1] == 2 else roc_auc_score(y_, proba, multi_class="ovr")),
    }

In [68]:
import time
from math import prod
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

search_space = {
    "Logistic Regression": (
        LogisticRegression(max_iter=400, random_state=SEED),
        {"C": [0.001, 0.01, 0.1, 1, 10, 100, 1000, 10000],
         "solver": ["lbfgs", "saga"]},
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"n_neighbors": [3, 5, 7, 9, 11, 15],
         "weights": ["uniform", "distance"],
         "p": [1, 2]},
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=SEED),
        {"max_depth": [2, 3, 5, 8, None],
         "min_samples_split": [2, 5, 10],
         "min_samples_leaf": [1, 2, 4]},
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=SEED, n_jobs=-1),
        {"n_estimators": [100, 300],
         "max_depth": [None, 5, 10],
         "min_samples_leaf": [1, 2, 4]},
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=SEED),
        {"n_estimators": [100, 200],
         "learning_rate": [0.01, 0.05, 0.1],
         "max_depth": [2, 3, 4]},
    ),
    "SVC": (
        SVC(probability=True, random_state=SEED),   # needed for predict_proba
        {"C": [0.1, 1, 10, 100],
         "kernel": ["linear", "rbf"],
         "gamma": ["scale", "auto"]},
    ),
    "XGBoost": (
        XGBClassifier(random_state=SEED, eval_metric="logloss"),
        {"n_estimators": [100, 300],
         "learning_rate": [0.01, 0.05, 0.1],
         "max_depth": [3, 5, 7]},
    ),
}

final_models = {}
results = {}

for name, (estimator, grid) in search_space.items():
    n_configs = prod(len(v) for v in grid.values())

    grid_s = GridSearchCV(estimator=estimator, param_grid=grid,
                          cv=5, scoring="accuracy", n_jobs=-1)

    t0 = time.time()
    grid_s.fit(X_train, y_train)
    print(f"{name}: {n_configs} configurations tried in {time.time()-t0:.1f}s")

    train_m = compute_metrics(grid_s, X_train, y_train)
    test_m  = compute_metrics(grid_s, X_test,  y_test)

    final_models[name] = grid_s
    results[name] = {
        "best_params": grid_s.best_params_,
        "n_configs": n_configs,
        "cv_accuracy": grid_s.best_score_,
        "train_accuracy": train_m["accuracy"],
        "test_accuracy": test_m["accuracy"],
        "train_f1_macro": train_m["f1_macro"],
        "test_f1_macro": test_m["f1_macro"],
        "train_auc": train_m["auc_ovr_macro"],
        "test_auc": test_m["auc_ovr_macro"],
    }

Logistic Regression: 16 configurations tried in 2.3s
KNN: 24 configurations tried in 0.2s
Decision Tree: 45 configurations tried in 0.2s
Random Forest: 18 configurations tried in 3.0s
Gradient Boosting: 18 configurations tried in 3.1s
SVC: 16 configurations tried in 46.1s
XGBoost: 18 configurations tried in 1.4s


In [69]:
#view the results
summary = pd.DataFrame(results).T
summary["acc_gap"] = summary["train_accuracy"] - summary["test_accuracy"]

num_cols = summary.columns.drop(["best_params", "n_configs"])
summary[num_cols] = summary[num_cols].astype(float).round(4)

summary.sort_values("test_accuracy", ascending=False)

,best_params,n_configs,cv_accuracy,train_accuracy,test_accuracy,train_f1_macro,test_f1_macro,train_auc,test_auc,acc_gap
Logistic Regression,"{'C': 10, 'solver': 'lbfgs'}",16,0.9538,0.9604,0.9649,0.9575,0.9623,0.9950,0.9957,-0.0045
SVC,"{'C': 1, 'gamma': 'scale', 'kernel': 'linear'}",16,0.9560,0.9582,0.9561,0.9552,0.9521,0.9952,0.9964,0.0021
XGBoost,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",18,0.9780,1.0000,0.9561,1.0000,0.9521,1.0000,0.9914,0.0439
KNN,"{'n_neighbors': 11, 'p': 1, 'weights': 'distan...",24,0.9429,1.0000,0.9474,1.0000,0.9429,1.0000,0.9734,0.0526
Random Forest,"{'max_depth': None, 'min_samples_leaf': 1, 'n_...",18,0.9582,1.0000,0.9474,1.0000,0.9435,1.0000,0.9937,0.0526
Gradient Boosting,"{'learning_rate': 0.1, 'max_depth': 2, 'n_esti...",18,0.9714,1.0000,0.9474,1.0000,0.9429,1.0000,0.9914,0.0526
Decision Tree,"{'max_depth': 5, 'min_samples_leaf': 1, 'min_s...",45,0.9385,0.9912,0.9211,0.9906,0.9164,0.9971,0.9163,0.0702
